In [ ]:
# 01 — Exploratory Data Analysis. The "look before you model" step. Refreshes
# the feature table from Supabase, caches it locally, and inspects
# it: data coverage per station, the PM2.5 distribution (heavy-tailed), the
# daily/weekly pollution cycles, how strongly past values predict future ones,
# and a preview of the time-based train/val/test split.

In [ ]:
# Cell 1 — refresh the training feature snapshot from Supabase
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'common').exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common.postgres_io import fetch_df

CACHE = ROOT / 'modeling' / 'feat_airquality.parquet'
df = fetch_df('''
    SELECT *
    FROM analytics.feat_airquality
    ORDER BY station_id, valid_time
''')
df.columns = df.columns.str.lower()
df['valid_time'] = pd.to_datetime(df['valid_time'], utc=True)
df = df.sort_values(['station_id', 'valid_time']).reset_index(drop=True)
df.to_parquet(CACHE, index=False)

print('saved fresh Supabase data:', CACHE)
print('rows:', len(df))
print('range:', df.valid_time.min(), '->', df.valid_time.max())
print(df.groupby('station_id').size())
df.head()

In [ ]:
# Cell 2 — coverage, depth, credits/seasonality sanity
print("range:", df.valid_time.min(), "->", df.valid_time.max(),
      "| span days:", (df.valid_time.max() - df.valid_time.min()).days)
print("\nrows per station:\n", df.station_id.value_counts())
print("\nmonths present (climatology depth):\n", df.valid_time.dt.to_period("M").value_counts().sort_index())
print("\nimputed fraction per station:\n", df.groupby("station_id").was_imputed.mean().round(3))

In [ ]:
# Cell 3 — target distribution (PM2.5 is heavy-tailed; this decides log-target vs raw)
print(df.groupby("station_id").pm25.describe()[["mean","50%","max"]])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.pm25.hist(bins=60, ax=ax[0]); ax[0].set_title("PM2.5 (µg/m³)")
np.log1p(df.pm25).hist(bins=60, ax=ax[1]); ax[1].set_title("log1p(PM2.5)")
plt.tight_layout()

In [ ]:
# Cell 4 — seasonality: the diurnal cycle (your local-time feature) + weekly
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby("hour_local").pm25.mean().plot(ax=ax[0], marker="o", title="mean PM2.5 by local hour")
df.groupby("day_of_week").pm25.mean().plot(ax=ax[1], marker="o", title="mean PM2.5 by day_of_week")
plt.tight_layout()

In [ ]:
# Cell 5 — eyeball the series per station (spot smoke spikes / sensor dropouts)
fig, ax = plt.subplots(figsize=(12, 4))
for sid, g in df.groupby("station_id"):
    ax.plot(g.valid_time, g.pm25, lw=0.8, label=sid)
ax.legend(); ax.set_ylabel("µg/m³"); ax.set_title("PM2.5 over time")

In [ ]:
# Cell 6 — persistence strength: how well do lags predict? (sets the baseline bar)
print(df[["pm25","pm25_lag_1h","pm25_lag_24h","pm25_lag_48h"]].corr()["pm25"].round(3))

In [ ]:
# Cell 7 — build the t+24h target the GAP-AWARE way (previews 2b's #1 silent bug)
H = 24
df["target_pm25"] = df.groupby("station_id").pm25.shift(-H)
ahead = df.groupby("station_id").valid_time.shift(-H) - df.valid_time
df["target_valid"] = ahead == pd.Timedelta(hours=H)   # reject pairs that straddle a dropped gap
print("rows:", len(df), "| usable t+24h examples:", int(df.target_valid.sum()))
print(df[df.target_valid].station_id.value_counts())

In [ ]:
# Cell 8 — exceedance class balance (decides pos_weight + whether binary is even viable now)
THRESH = 35.4
sub = df[df.target_valid]
exc = sub.target_pm25 > THRESH
print(f"exceedance (>{THRESH}) base rate: {exc.mean():.3%}  ({int(exc.sum())}/{len(sub)})")
print(sub.assign(e=exc).groupby("station_id").e.mean().round(4))

In [ ]:
# Cell 9 — feature → target correlation preview (what the LSTM/XGB will lean on)
feat = ["pm25_lag_1h","pm25_lag_24h","pm25_lag_48h","pm25_roll_mean_24h","pm25_roll_max_24h",
        "temperature_2m","wind_speed_10m","wind_direction_10m","precipitation","hour_local","day_of_week"]
print(sub[feat + ["target_pm25"]].corr()["target_pm25"].drop("target_pm25").sort_values().round(3))

In [ ]:
# Cell 10 — time-aware split preview (70/15/15 BY TIME — sizes your future test set)
t = df.valid_time; q70, q85 = t.quantile(.70), t.quantile(.85)
for name, m in [("train", t<q70), ("val",(t>=q70)&(t<q85)), ("test", t>=q85)]:
    s = df[m & df.target_valid]
    print(f"{name:6} n={len(s):5d}  exceedance={(s.target_pm25>35.4).mean():.2%}  ({s.valid_time.min()} → {s.valid_time.max()})")